In [14]:
# ---------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import country_converter as coco
import numpy as np
import time
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox
import re
from uncertainties import unumpy as unp


# ---------------------------------------------------------------------
#  File paths
# ---------------------------------------------------------------------
file_path_main = "../results/Scale_up_output_MS.pkl"
file_path_percent = "../results/Scale_up_PERCENT_ECW_POLL_MS.pkl"
file_path_cr_man = "../results/Scale_up_CR_MAN_MS.pkl"
file_path_cr_repur = "../results/Scale_up_CR_REPUR_MS.pkl"
file_path_coalbag = "../results/Scale_up_COALBAG_MS.pkl"
file_path_cr_stock = "../results/Scale_up_CR_STOCK.pkl"
file_path_ecw_country = "../results/ECWbyCountry.csv"

# ---------------------------------------------------------------------
#  Functions Import
# ---------------------------------------------------------------------
from functions_MS import get_series, get_ECW


CADRPP = 100 # L/s
UNRegion_list = ['Australia and New Zealand',
 'Caribbean',
 'Central America',
 'Central Asia',
 'Eastern Africa',
 'Eastern Asia',
 'Eastern Europe',
 'Melanesia',
 'Micronesia',
 'Middle Africa',
 'Northern Africa',
 'Northern America',
 'Northern Europe',
 'Polynesia',
 'South America',
 'South-eastern Asia',
 'Southern Africa',
 'Southern Asia',
 'Southern Europe',
 'Western Africa',
 'Western Asia',
 'Western Europe']

# ---------------------------------------------------------------------
#  Load CSVs (main airflow + percentage + subtypes)
# ---------------------------------------------------------------------
def load_dataset(path):
    df = pd.read_pickle(path)
    week_cols = df.columns[3:]
    df_filtered = df.loc[UNRegion_list]
    df_nominal = df_filtered.map(
        lambda x: unp.nominal_values(x) if hasattr(x, "nominal_value") else x
    )   
    df_nominal = df_nominal.drop(columns=[df.columns[0], df.columns[1]]) 
    return df_nominal, week_cols

# Load datasets
df_main, week_cols = load_dataset(file_path_main)
df_percent, _ = load_dataset(file_path_percent)
df_cr_man, _ = load_dataset(file_path_cr_man)
df_cr_repur, _ = load_dataset(file_path_cr_repur)
df_coalbag, _ = load_dataset(file_path_coalbag)
df_cr_stock, _ = load_dataset(file_path_cr_stock)

# Load ECW lower/upper bounds
df_ecw = pd.read_pickle(file_path_main)
df_ecw = df_ecw.loc[UNRegion_list]
df_ecw = df_ecw.iloc[:,:2]
# print(df_ecw)
ecw_lower_dict={}
ecw_upper_dict = {}
for region in UNRegion_list:
    row = df_ecw.loc[region]
    ecw_upper_dict[region] = CADRPP*row[1]
    ecw_lower_dict[region] = CADRPP*row[0]

# ---------------------------------------------------------------------
#  debug++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ---------------------------------------------------------------------

def debug_preview(df, name, n=5):
    """Print a detailed preview for each dataset"""
    print(f"\n🔍 [DEBUG] {name}")
    print(f"Shape: {df.shape}")
    print("Columns:", list(df.columns[:10]), "...")
    print("Index (first 5):", list(df.index[:5]))
    print(f"\nSample (first {n} rows):")
    print(df.head(n))
    print("\nRandom Sample:")
    print(df.sample(min(n, len(df))))
    print("\nNaN Summary (count per column, top 10):")
    print(df.isna().sum().head(10))
    print("-" * 80)


# --- Run the preview for all key datasets ---
debug_preview(df_main, "Main Airflow Data")
debug_preview(df_percent, "ECW Coverage Percentage Data")
debug_preview(df_cr_man, "CR Box Manufacturing Data")
debug_preview(df_cr_repur, "CR Box Repurposing Data")
debug_preview(df_coalbag, "Coalbaghouse Data")
debug_preview(df_cr_stock, "CR Box Stock Data")

# --- ECW Lower/Upper Bound Dictionaries ---
print("\n📊 [DEBUG] ECW Bound Dictionaries (5 random samples):")
for c, (low, high) in list(zip(ecw_lower_dict.keys(), zip(ecw_lower_dict.values(), ecw_upper_dict.values())))[:10]:
    print(f"  {c:<25} | ECW ILO = {low:<12} | ECW Poll = {high:<12}")

# --- Week columns sanity check ---
print("\n🕓 [DEBUG] Week Columns Detected:")
print(list(week_cols[:15]), "...")
print(f"Total week columns: {len(week_cols)}")

# --- Check if any UN region data is missing ---
missing_regions = [r for r in UNRegion_list if r not in df_main.index]
if missing_regions:
    print("\n⚠️ [WARNING] Missing UN Regions from df_main:")
    print(missing_regions)
else:
    print("\n✅ All UN regions found in df_main.")




🔍 [DEBUG] Main Airflow Data
Shape: (22, 53)
Columns: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11] ...
Index (first 5): ['Australia and New Zealand', 'Caribbean', 'Central America', 'Central Asia', 'Eastern Africa']

Sample (first 5 rows):
                           2              3             4             5   \
Australia and New Zealand   0  209322.123229  4.709748e+05  7.849580e+05   
Caribbean                   0  358522.276117  7.364529e+05  1.133792e+06   
Central America             0  775655.980373  1.722444e+06  2.840363e+06   
Central Asia                0  106295.106045  2.391640e+05  3.986066e+05   
Eastern Africa              0       0.000000  0.000000e+00  0.000000e+00   

                                     6             7             8   \
Australia and New Zealand  1.039714e+07  1.268323e+07  1.564963e+07   
Caribbean                  2.090631e+06  2.623828e+06  3.409332e+06   
Central America            1.152182e+07  1.888744e+07  2.847778e+07   
Central Asia               6.30

In [17]:
# ---------------------------------------------------------------------
#  Widgets
# ---------------------------------------------------------------------
map_choice = widgets.Dropdown(
    options=[
        "ALL",
        "ECW Coverage %",
        "CR Box Manufacturing",
        "CR Box Repurposing",
        "Coalbaghouse",
        "CR Box Stock",
    ],
    value="ALL",
    description="Type:",
    style={"description_width": "initial"},
)

region_filter = widgets.Dropdown(
    options=["All"] + sorted(UNRegion_list),
    value="All",
    description="UN Region:",
    style={"description_width": "initial"},
)

region_select = widgets.SelectMultiple(
    options=sorted(UNRegion_list),
    value=["Eastern Asia"],
    description="Regions:",
    layout=widgets.Layout(width="50%", height="150px"),
)

week_index = widgets.IntSlider(
    value=1,
    min=1,
    max=len(week_cols),
    step=1,
    description="Week:",
    layout=widgets.Layout(width="80%"),
)

play_button = widgets.Button(
    description="▶ Play",
    tooltip="Auto-play weeks",
    button_style="success",
    layout=widgets.Layout(width="100px", height="35px"),
)

out = widgets.Output()

# ---------------------------------------------------------------------
#  Build region → ISO3 mapping
# ---------------------------------------------------------------------
cc = coco.CountryConverter()
region_to_iso = {}

for region in UNRegion_list:
    matches = cc.data.loc[cc.data["UNregion"] == region, "ISO3"].tolist()
    if not matches:
        matches = cc.data.loc[
            cc.data["UNregion"].str.contains(region.split()[0], case=False, na=False),
            "ISO3",
        ].tolist()
    region_to_iso[region] = matches

print(f"[DEBUG] Region→ISO mapping built ({len(region_to_iso)} regions):")
for k, v in list(region_to_iso.items())[:5]:
    print(f"  {k}: {v[:5]} ...")
print("✅ Mapping ready.\n")

# ---------------------------------------------------------------------
#  Plot update function
# ---------------------------------------------------------------------
def update_plots(change=None):
    week = str(week_index.value)
    map_val = map_choice.value
    filter_value = region_filter.value

    with out:
        clear_output(wait=True)

        # ---------- Choose dataset ----------
        if map_val == "ECW Coverage %":
            df_map = df_percent.copy()
            vmin, vmax = 0, 1
            color_scale = [(0, "white"), (1, "orange")]
            title_text = f"ECW Coverage (%) by Country (Week {week})"

        elif map_val == "CR Box Manufacturing":
            df_map = df_cr_man.copy()
            vmin, vmax = df_map.min().min(), df_map.max().max()
            color_scale = [(0, "white"), (1, "orange")]
            title_text = f"CR Box Manufacturing (Week {week})"

        elif map_val == "CR Box Repurposing":
            df_map = df_cr_repur.copy()
            vmin, vmax = df_map.min().min(), df_map.max().max()
            color_scale = [(0, "white"), (1, "orange")]
            title_text = f"CR Box Repurposing (Week {week})"

        elif map_val == "Coalbaghouse":
            df_map = df_coalbag.copy()
            vmin, vmax = df_map.min().min(), df_map.max().max()
            color_scale = [(0, "white"), (1, "orange")]
            title_text = f"Coalbaghouse (Week {week})"

        elif map_val == "CR Box Stock":
            df_map = df_cr_stock.copy()
            vmin, vmax = df_map.min().min(), df_map.max().max()
            color_scale = [(0, "white"), (1, "orange")]
            title_text = f"CR Box Stock (Week {week})"

        else:
            df_map = df_main.copy()
            vmin, vmax = df_map.min().min(), df_map.max().max()
            color_scale = [(0, "white"), (1, "darkgreen")]
            title_text = f"Total Air Flow by Country (up to Week {week})"

        # ---------- Convert region data to country-level ----------
        country_values = []
        for region_name, row in df_map.iterrows():
            if region_name not in region_to_iso:
                continue
            try:
                value = row[int(week)]
            except Exception:
                continue

            for iso3 in region_to_iso[region_name]:
                country_values.append(
                    {"ISO3": iso3, "Region": region_name, week: value}
                )

        choropleth_df = pd.DataFrame(country_values)
        print(f"[DEBUG] choropleth_df built: {len(choropleth_df)} rows")

        if choropleth_df.empty:
            print("[⚠️] No data found — check mapping or week index.")
        else:
            # ---------- Choropleth ----------
            fig1 = px.choropleth(
                choropleth_df,
                locations="ISO3",
                locationmode="ISO-3",
                color=week,
                hover_name="Region",
                color_continuous_scale=color_scale,
                range_color=[vmin, vmax],
                title=title_text,
            )
            fig1.update_layout(margin=dict(l=0, r=0, t=50, b=0))
            fig1.show()

        # --- Week slider + play button
        control_row = HBox(
            [week_index, play_button],
            layout=widgets.Layout(
                justify_content="center", align_items="center", margin="10px 0px"
            ),
        )
        display(control_row)

        # ---------- Second chart: Airflow + ECW Range ----------
        fig, ax = plt.subplots(figsize=(10, 6))
        if map_val == "ALL":
            df_used = df_main
        elif map_val == "CR Box Manufacturing":
            df_used = df_cr_man
        elif map_val == "CR Box Repurposing":
            df_used = df_cr_repur
        elif map_val == "Coalbaghouse":
            df_used = df_coalbag
        elif map_val == "CR Box Stock":
            df_used = df_cr_stock
        elif map_val == "ECW Coverage %":
            df_used = df_percent
        else:
            df_used = df_main

        for region in region_select.value:
            if region not in df_used.index:
                continue
            mean_series = df_used.loc[region].astype(float)
            t = np.arange(len(mean_series))
            ax.plot(t, mean_series, label=f"{region} ({map_val})", linewidth=2)

            # ECW bounds (mean across region)
            countries = region_to_iso.get(region, [])
            ecw_lowers = ecw_lower_dict[region]
            ecw_uppers = ecw_upper_dict[region]

 
            ax.axhspan(ecw_lowers, ecw_uppers, color="green", alpha=0.15)


        ax.set_title(f"{map_val} Air Flow vs ECW Range by Selected UN Regions")
        ax.set_xlabel("Weeks")
        ax.set_ylabel("Air Flow")
        ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
        plt.show()


# ---------------------------------------------------------------------
#  Play logic
# ---------------------------------------------------------------------
def play_clicked(b):
    for w in range(1, len(week_cols) + 1):
        week_index.value = w
        time.sleep(0.5)


play_button.on_click(play_clicked)
week_index.observe(update_plots, names="value")
region_filter.observe(update_plots, names="value")
region_select.observe(update_plots, names="value")
map_choice.observe(update_plots, names="value")

# ---------------------------------------------------------------------
#  Style + Layout
# ---------------------------------------------------------------------
custom_style = """
<style>
#custom-container {
    background-color: #fffacd;
    padding: 15px;
    border-radius: 10px;
    margin-top: 10px;
    margin-bottom: 20px;
}
#custom-banner {
    width: 100%;
    background-color: #006400;
    color: white;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    padding: 10px;
    border-radius: 6px;
    margin-bottom: 15px;
}
</style>
"""
display(HTML(custom_style))

container = VBox(
    [
        widgets.HTML('<div id="custom-banner">ALLFED</div>'),
        HBox([region_filter, map_choice]),
        region_select,
        out,
    ]
)
container.layout = widgets.Layout(
    width="100%",
    padding="15px",
    border="solid 2px #006400",
    border_radius="10px",
    background_color="#fffacd",
)
display(container)
update_plots()



[DEBUG] Region→ISO mapping built (22 regions):
  Australia and New Zealand: ['AUS', 'CXR', 'CCK', 'HMD', 'NZL'] ...
  Caribbean: ['AIA', 'ATG', 'ABW', 'BHS', 'BRB'] ...
  Central America: ['BLZ', 'CRI', 'SLV', 'GTM', 'HND'] ...
  Central Asia: ['KAZ', 'KGZ', 'TJK', 'TKM', 'UZB'] ...
  Eastern Africa: ['IOT', 'BDI', 'COM', 'DJI', 'ERI'] ...
✅ Mapping ready.



In [16]:
import country_converter as coco
cc = coco.CountryConverter()

print("=== Unique values in cc.data['UNregion'] ===")
print(cc.data['UNregion'].dropna().unique())
print("\n=== Unique values in cc.data['continent'] ===")
print(cc.data['continent'].dropna().unique())


=== Unique values in cc.data['UNregion'] ===
['Southern Asia' 'Northern Europe' 'Southern Europe' 'Northern Africa'
 'Polynesia' 'Middle Africa' 'Caribbean' 'Antarctica' 'South America'
 'Western Asia' 'Australia and New Zealand' 'Western Europe'
 'Eastern Europe' 'Central America' 'Western Africa' 'Northern America'
 'Southern Africa' 'Eastern Africa' 'South-eastern Asia' 'Eastern Asia'
 'Melanesia' 'Micronesia' 'Central Asia']

=== Unique values in cc.data['continent'] ===
['Asia' 'Europe' 'Africa' 'Oceania' 'America' 'Antarctica']
